In [1]:
pip list

Package                           Version
--------------------------------- -------------------
absl-py                           2.0.0
aiobotocore                       2.25.0
aiohappyeyeballs                  2.6.1
aiohttp                           3.11.10
aioitertools                      0.7.1
aiosignal                         1.2.0
alabaster                         1.0.0
altair                            6.0.0
anaconda-anon-usage               0.7.5
anaconda-auth                     0.13.0
anaconda-catalogs                 0.2.0
anaconda-cli-base                 0.8.1
anaconda-client                   1.14.1
anaconda-navigator                2.7.0
anaconda-project                  0.12.0
annotated-types                   0.6.0
anyio                             4.12.1
appdirs                           1.4.4
archspec                          0.2.3
argon2-cffi                       21.3.0
argon2-cffi-bindings              25.1.0
arrow                             1.4.0
astroid        

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
# AGG - Anti Grain Geometry engine - we will not print interactive plots instead we render 
# directly to a image file (like PNG, JPG)
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import os

In [4]:
OUTPUT = "c://Users//Kiran//Comcast_MachineLearning//outputs"
os.makedirs(OUTPUT, exist_ok=True)

## Generate Dataset

In [12]:
np.random.seed(42)
# data set count = 500 records
N = 500

# gender should be male or female for 500 records with  male - 48% and female - 52%
genders = np.random.choice(['Male', 'Female'], N, p=[0.48, 0.52])

# fill the race_groups column with these 5 values
groups = np.random.choice(["A", "B", "C", "D", "E"], N)

# parent_education column is filled with the following data
edu = np.random.choice(
    ["high school", "some college", "associate's", "bachelor's", "master's"],
    N, p=[0.18, 0.24, 0.22, 0.26, 0.10]
)

# lunch for the students
lunch = np.random.choice(["standard", "free/reduced"], N, p=[0.65, 0.35])

test_prep = np.random.choice(["none", "completed"], N ,p=[0.60, 0.40])

edu_boost = {"high school":0, "some college":3, "associate's": 5, "bachelor's": 9, "master's": 14}
prep_boost = {"none": 0, "completed":8}

base_math = (55 + np.array([edu_boost[e] for e in edu])
                + np.array([prep_boost[t] for t in test_prep])
                + np.random.normal(0, 12, N))

base_read = base_math * 0.75 + np.random.normal(10, 8, N)
base_write = base_math * 0.72 + np.random.normal(12, 8, N)

math_score = np.clip(base_math, 0, 100).astype(float)
read_score = np.clip(base_read, 0, 100).astype(float)
write_score = np.clip(base_write, 0, 100).astype(float)

# inject 5% of missing values in 3 cols
for arr in [math_score, read_score, write_score]:
    idx = np.random.choice(N, int(N * 0.05), replace=False)
    arr[idx] = np.nan

# inject few outliers
math_score[np.random.choice(np.where(~np.isnan(math_score))[0], 8)] = np.random.choice([2, 
                                                        3, 4, 96, 97, 98, 99], 8)

df = pd.DataFrame( {
    "gender"          : genders, 
    "race_group"      : groups, 
    "parent_education": edu,
    "lunch"            : lunch,
    "test_prep"       : test_prep,
    "math_score"      : math_score, 
    "reading_score"   : read_score, 
    "wrting_score"    : write_score,   
})

print(f"Dataset Generated: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Numeric Columns  : {list(df.select_dtypes('number').columns)}")
print(f"Categorical      : {list(df.select_dtypes('object').columns)}")

Dataset Generated: 500 rows x 8 columns
Numeric Columns  : ['math_score', 'reading_score', 'wrting_score']
Categorical      : ['gender', 'race_group', 'parent_education', 'luch', 'test_prep']


## Data Overview

In [15]:
print("df.shape".center(60, "-"))
print(df.shape)

print("df.types".center(60, "-"))
print(df.dtypes)

print("df.head".center(60, "-"))
print(df.head(5).to_string())

print("df.describe".center(60, "-"))
print(df.describe().round(2).to_string)

print("df.describe(include='object'".center(60, "-"))
print(df.describe(include='object').to_string())

print("gender value count".center(60, "-"))
print(df['gender'].value_counts())

df.to_csv(OUTPUT + "/data.csv")

--------------------------df.shape--------------------------
(500, 8)
--------------------------df.types--------------------------
gender                  str
race_group              str
parent_education        str
luch                    str
test_prep               str
math_score          float64
reading_score       float64
wrting_score        float64
dtype: object
--------------------------df.head---------------------------
   gender race_group parent_education          luch  test_prep  math_score  reading_score  wrting_score
0    Male          A     some college      standard  completed   84.175643      79.073732     67.425406
1  Female          A       bachelor's  free/reduced  completed   96.428140      93.239495     72.440028
2  Female          A     some college      standard  completed   66.335417      74.725546           NaN
3  Female          E      high school      standard       none   60.018245      58.927731     53.674744
4    Male          D       bachelor's      standar

## Misssing value Analysis

In [19]:

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PURPLE = "#534AB7"; TEAL = "#0F6E56"; BLUE = "#185FA5"
CORAL  = "#D85A30"; AMBER= "#854F0B"; GREEN= "#3B6D11"
PAL    = [PURPLE, TEAL, BLUE, CORAL, AMBER, GREEN]

def save(name):
    plt.savefig(f"{OUTPUT}/{name}", dpi=130, bbox_inches="tight")
    plt.close()
    print(f"     Saved → {name}")

# check for nan values and return the count of missing values per column
missing = df.isnull().sum()

# convert the counts into percentage of total rows with decimals rounded to 2
missing_pct = (missing / len(df) * 100).round(2)

# build summary dataframe with counts and percentage and filter all cols having missing values
miss_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
miss_df = miss_df[miss_df["Missing Count"] > 0]

# printing the missing values in a neat table
print("\n── Missing values ──")
print(miss_df.to_string())

# Visual: missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Stage 3 — Missing Value Analysis", fontsize=14,
             fontweight="bold", y=1.01)

# Bar chart of missing %
ax = axes[0]
miss_df["Missing %"].plot(kind="bar", ax=ax, color=CORAL,
                          edgecolor="white", width=0.55)
ax.set_title("Missing value percentage per column", fontsize=12)
ax.set_ylabel("Missing %")
ax.set_xlabel("")
ax.set_ylim(0, 15)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{bar.get_height():.1f}%", ha="center", fontsize=9)
sns.despine(ax=ax)

# Heatmap of nulls
ax = axes[1]
null_map = df.isnull().astype(int)
sns.heatmap(null_map.T, ax=ax, cmap=["#E1F5EE", CORAL],
            cbar=False, yticklabels=True, xticklabels=False)
ax.set_title("Null pattern heatmap (red = missing)", fontsize=12)
ax.set_ylabel("")

plt.tight_layout()
save("03_missing_values.png")

# Handle missing values
print("\n── Handling missing values with median imputation ──")
for col in ["math_score", "reading_score", "wrting_score"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: filled with median = {median_val:.1f}")

print(f"\n  Remaining nulls: {df.isnull().sum().sum()}")


── Missing values ──
              Missing Count  Missing %
wrting_score             25        5.0
     Saved → 03_missing_values.png

── Handling missing values with median imputation ──
  math_score: filled with median = 64.0
  reading_score: filled with median = 59.7
  wrting_score: filled with median = 57.7

  Remaining nulls: 0


## Univariate analysis

In [22]:
num_cols = ["math_score", "reading_score", "wrting_score"]
cat_cols = ["gender", "race_group", "parent_education",
            "luch", "test_prep"]

# ── 4a: Numeric distributions ──────────────────────────────────
print("\n── Numeric: descriptive stats ──")
for col in num_cols:
    s = df[col]
    print(f"\n  {col}:")
    print(f"    Mean={s.mean():.2f}  Median={s.median():.2f}  "
          f"Std={s.std():.2f}")
    print(f"    Skewness={s.skew():.3f}  Kurtosis={s.kurtosis():.3f}")
    print(f"    Min={s.min():.1f}  Max={s.max():.1f}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Stage 4 — Univariate Analysis: Numeric Scores",
             fontsize=14, fontweight="bold", y=1.01)

colors = [PURPLE, TEAL, BLUE]
for i, (col, c) in enumerate(zip(num_cols, colors)):
    # Histogram + KDE
    ax = axes[0][i]
    ax.hist(df[col], bins=30, color=c, alpha=0.75,
            edgecolor="white", linewidth=0.4, density=True)
    df[col].plot(kind="kde", ax=ax, color="black", linewidth=1.4)
    ax.axvline(df[col].mean(),   color=CORAL, linewidth=1.4,
               linestyle="--", label=f"Mean={df[col].mean():.1f}")
    ax.axvline(df[col].median(), color=AMBER, linewidth=1.4,
               linestyle=":",  label=f"Median={df[col].median():.1f}")
    ax.set_title(col.replace("_"," ").title(), fontsize=11)
    ax.set_xlabel("Score")
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

    # Box plot
    ax = axes[1][i]
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=c, alpha=0.6),
               medianprops=dict(color="black", linewidth=2),
               whiskerprops=dict(linewidth=1.2),
               capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markerfacecolor=CORAL,
                               markersize=4, alpha=0.6))
    ax.set_title(f"{col.replace('_',' ').title()} — boxplot", fontsize=11)
    ax.set_ylabel("Score")
    ax.set_xticks([])
    sns.despine(ax=ax)

plt.tight_layout()
save("04a_univariate_numeric.png")

# ── 4b: Categorical distributions ──────────────────────────────
print("\n── Categorical: value counts ──")
for col in cat_cols:
    print(f"\n  {col}:")
    print(df[col].value_counts().to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Stage 4 — Univariate Analysis: Categorical Features",
             fontsize=14, fontweight="bold", y=1.01)

for i, col in enumerate(cat_cols):
    ax = axes[i//3][i%3]
    vc = df[col].value_counts()
    bars = ax.bar(range(len(vc)), vc.values,
                  color=PAL[:len(vc)], edgecolor="white", linewidth=0.4)
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=20, ha="right", fontsize=9)
    ax.set_title(col.replace("_"," ").title(), fontsize=11)
    ax.set_ylabel("Count")
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                str(val), ha="center", fontsize=9)
    sns.despine(ax=ax)

axes[1][2].set_visible(False)
plt.tight_layout()
save("04b_univariate_categorical.png")


── Numeric: descriptive stats ──

  math_score:
    Mean=64.77  Median=64.04  Std=13.83
    Skewness=-0.088  Kurtosis=1.382
    Min=2.0  Max=100.0

  reading_score:
    Mean=59.45  Median=59.68  Std=12.72
    Skewness=0.080  Kurtosis=0.424
    Min=19.7  Max=99.9

  wrting_score:
    Mean=57.97  Median=57.71  Std=11.91
    Skewness=0.062  Kurtosis=-0.191
    Min=25.0  Max=89.0
     Saved → 04a_univariate_numeric.png

── Categorical: value counts ──

  gender:
gender
Female    267
Male      233

  race_group:
race_group
A    109
E    105
C    105
B     97
D     84

  parent_education:
parent_education
bachelor's      131
associate's     124
some college    103
high school      92
master's         50

  luch:
luch
standard        337
free/reduced    163

  test_prep:
test_prep
none         301
completed    199
     Saved → 04b_univariate_categorical.png


## Bivariate Analysis

In [25]:
print("\n── Numeric vs Numeric: Pearson correlations ──")
corr = df[num_cols].corr()
print(corr.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Stage 5a — Numeric vs Numeric: Scatter plots",
             fontsize=14, fontweight="bold", y=1.01)

pairs = [("math_score","reading_score"),
         ("math_score","wrting_score"),
         ("reading_score","wrting_score")]

for ax, (x, y) in zip(axes, pairs):
    ax.scatter(df[x], df[y], alpha=0.3, s=18, color=PURPLE,
               rasterized=True)
    m, b, r, p, _ = stats.linregress(df[x], df[y])
    xr = np.linspace(df[x].min(), df[x].max(), 200)
    ax.plot(xr, m*xr+b, color=CORAL, linewidth=1.8,
            label=f"r = {r:.3f}")
    ax.set_xlabel(x.replace("_"," ").title())
    ax.set_ylabel(y.replace("_"," ").title())
    ax.set_title(f"{x.split('_')[0].title()} vs {y.split('_')[0].title()}")
    ax.legend(fontsize=10)
    sns.despine(ax=ax)

plt.tight_layout()
save("05a_bivariate_numeric.png")

# ── 5b: Categorical vs Numeric ──────────────────────────────────
print("\n── Categorical vs Numeric: group means ──")
for cat in ["gender","luch","test_prep"]:
    print(f"\n  {cat}:")
    print(df.groupby(cat)[num_cols].mean().round(2).to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Stage 5b — Categorical vs Numeric: Box plots",
             fontsize=14, fontweight="bold", y=1.01)

combos = [("gender","math_score"), ("luch","math_score"),
          ("test_prep","reading_score"),
          ("gender","reading_score"), ("luch","wrting_score"),
          ("test_prep","wrting_score")]

for ax, (cat, num) in zip(axes.flat, combos):
    groups_u = df[cat].unique()
    data  = [df[df[cat]==g][num].values for g in groups_u]
    bplot = ax.boxplot(data, patch_artist=True,
                       medianprops=dict(color="black",linewidth=2),
                       whiskerprops=dict(linewidth=1.2),
                       capprops=dict(linewidth=1.2),
                       flierprops=dict(marker="o",markerfacecolor=CORAL,
                                       markersize=3,alpha=0.5))
    for patch, c in zip(bplot["boxes"], PAL):
        patch.set_facecolor(c); patch.set_alpha(0.65)
    ax.set_xticklabels(groups_u, fontsize=9)
    ax.set_title(f"{cat.replace('_',' ').title()} vs "
                 f"{num.replace('_',' ').title()}", fontsize=10)
    ax.set_ylabel("Score")
    sns.despine(ax=ax)

plt.tight_layout()
save("05b_bivariate_cat_num.png")

# ── 5c: Categorical vs Categorical (crosstab heatmap) ────────────
print("\n── Categorical vs Categorical: crosstab ──")
ct = pd.crosstab(df["gender"], df["parent_education"],
                 normalize="index").round(3)
print(ct.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Stage 5c — Categorical vs Categorical",
             fontsize=14, fontweight="bold", y=1.01)

# Crosstab heatmap
ax = axes[0]
ct2 = pd.crosstab(df["gender"], df["parent_education"])
sns.heatmap(ct2, annot=True, fmt="d", cmap="Blues", ax=ax,
            linewidths=0.5, cbar_kws={"shrink":0.8})
ax.set_title("Gender × Parent Education (counts)", fontsize=11)
ax.set_ylabel("Gender"); ax.set_xlabel("Parent Education")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right",
                   fontsize=9)

# Grouped bar chart
ax = axes[1]
ct3 = pd.crosstab(df["test_prep"], df["luch"])
ct3.plot(kind="bar", ax=ax, color=PAL[:2], edgecolor="white",
         width=0.55)
ax.set_title("Test Prep × Lunch type (counts)", fontsize=11)
ax.set_xlabel("Test Preparation"); ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Lunch", fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
save("05c_bivariate_cat_cat.png")


── Numeric vs Numeric: Pearson correlations ──
               math_score  reading_score  wrting_score
math_score          1.000          0.680         0.687
reading_score       0.680          1.000         0.562
wrting_score        0.687          0.562         1.000
     Saved → 05a_bivariate_numeric.png

── Categorical vs Numeric: group means ──

  gender:
        math_score  reading_score  wrting_score
gender                                         
Female       65.50          60.21         58.56
Male         63.93          58.59         57.30

  luch:
              math_score  reading_score  wrting_score
luch                                                 
free/reduced       65.26          59.22         58.46
standard           64.53          59.57         57.74

  test_prep:
           math_score  reading_score  wrting_score
test_prep                                         
completed       70.01          63.13         61.39
none            61.30          57.03         55.71
    

## Multivariate Analysis

In [26]:
fig, ax = plt.subplots(figsize=(7, 5))
corr_full = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_full, dtype=bool))
cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(corr_full, mask=mask, cmap=cmap, annot=True, fmt=".3f",
            linewidths=0.5, ax=ax, annot_kws={"size":11},
            vmin=-1, vmax=1)
ax.set_title("Stage 6a — Correlation Heatmap",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
save("06a_correlation_heatmap.png")

# ── 6b: Pairplot (seaborn) ────────────────────────────────────────
print("  Generating pairplot (takes a moment)...")
pair_df = df[num_cols + ["gender"]].copy()
g = sns.pairplot(pair_df, hue="gender", diag_kind="kde",
                 plot_kws={"alpha":0.4, "s":20},
                 palette={"Male":BLUE,"Female":CORAL})
g.fig.suptitle("Stage 6b — Pairplot (scores by gender)",
               fontsize=13, fontweight="bold", y=1.01)
g.fig.savefig(f"{OUTPUT}/06b_pairplot.png", dpi=120,
              bbox_inches="tight")
plt.close()
print(f"     Saved → 06b_pairplot.png")

# ── 6c: Multi-group comparison ───────────────────────────────────
edu_order = ["high school","some college","associate's",
             "bachelor's","master's"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Stage 6c — Score by Parent Education & Test Prep",
             fontsize=13, fontweight="bold", y=1.01)

for ax, col in zip(axes, num_cols):
    means = df.groupby(["parent_education","test_prep"])[col]\
               .mean().unstack()
    x = np.arange(len(edu_order))
    w = 0.35
    for i, prep in enumerate(["none","completed"]):
        vals = [means.loc[e, prep] if e in means.index else 0
                for e in edu_order]
        bars = ax.bar(x + i*w - w/2, vals, w,
                      label=f"Prep: {prep}",
                      color=[PURPLE, TEAL][i],
                      edgecolor="white", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([e.replace("'","'") for e in edu_order],
                       rotation=20, ha="right", fontsize=8)
    ax.set_title(col.replace("_"," ").title(), fontsize=11)
    ax.set_ylabel("Mean score")
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

plt.tight_layout()
save("06c_multivariate_grouped.png")


     Saved → 06a_correlation_heatmap.png
  Generating pairplot (takes a moment)...
     Saved → 06b_pairplot.png
     Saved → 06c_multivariate_grouped.png


## Outliers Detection

In [27]:
print("\n── IQR method ──")
iqr_outliers = {}
for col in num_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask  = (df[col] < lower) | (df[col] > upper)
    iqr_outliers[col] = df[mask][col]
    print(f"  {col}: Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}  "
          f"bounds=[{lower:.1f}, {upper:.1f}]  "
          f"outliers={mask.sum()}")

print("\n── Z-score method (|z|>3) ──")
z_outliers = {}
for col in num_cols:
    z  = np.abs(stats.zscore(df[col]))
    mask = z > 3
    z_outliers[col] = df[mask][col]
    print(f"  {col}: {mask.sum()} outliers")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Stage 7 — Outlier Detection (IQR & Z-score)",
             fontsize=14, fontweight="bold", y=1.01)

colors = [PURPLE, TEAL, BLUE]
for i, (col, c) in enumerate(zip(num_cols, colors)):
    # IQR boxplot with outliers highlighted
    ax = axes[0][i]
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR; upper = Q3 + 1.5*IQR
    normal   = df[(df[col] >= lower) & (df[col] <= upper)][col]
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    ax.scatter([1]*len(normal),   normal,   alpha=0.25, s=15,
               color=c, label="normal")
    ax.scatter([1]*len(outliers), outliers, alpha=0.9,  s=40,
               color=CORAL, zorder=5, label=f"outliers ({len(outliers)})")
    ax.set_xlim(0.5, 1.5); ax.set_xticks([])
    ax.set_title(f"{col.replace('_',' ').title()} — IQR", fontsize=10)
    ax.set_ylabel("Score"); ax.legend(fontsize=9)
    sns.despine(ax=ax)

    # Z-score distribution with outlier zones
    ax = axes[1][i]
    z = stats.zscore(df[col])
    ax.hist(z, bins=30, color=c, alpha=0.7, edgecolor="white",
            linewidth=0.4)
    ax.axvline( 3, color=CORAL, linewidth=1.5, linestyle="--",
                label="|z|=3 threshold")
    ax.axvline(-3, color=CORAL, linewidth=1.5, linestyle="--")
    ax.set_title(f"{col.replace('_',' ').title()} — Z-score", fontsize=10)
    ax.set_xlabel("Z-score"); ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

plt.tight_layout()
save("07_outlier_detection.png")


── IQR method ──
  math_score: Q1=56.8  Q3=72.6  IQR=15.8  bounds=[33.2, 96.2]  outliers=18
  reading_score: Q1=51.6  Q3=67.3  IQR=15.6  bounds=[28.2, 90.7]  outliers=12
  wrting_score: Q1=50.0  Q3=66.3  IQR=16.3  bounds=[25.6, 90.7]  outliers=1

── Z-score method (|z|>3) ──
  math_score: 2 outliers
  reading_score: 2 outliers
  wrting_score: 0 outliers
     Saved → 07_outlier_detection.png


## Final Summary

In [30]:
print("\n── Creating derived features ──")
df["avg_score"]   = df[num_cols].mean(axis=1).round(2)
df["total_score"] = df[num_cols].sum(axis=1).round(1)
df["score_range"] = (df[num_cols].max(axis=1) -
                     df[num_cols].min(axis=1)).round(1)
df["grade"] = pd.cut(df["avg_score"],
                     bins=[0,40,55,70,85,101],
                     labels=["F","D","C","B","A"])

print(df[["avg_score","total_score","score_range","grade"]].head(8)
      .to_string())
print("\n── Grade distribution ──")
print(df["grade"].value_counts().sort_index().to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Stage 8 — Feature Engineering & EDA Summary",
             fontsize=14, fontweight="bold", y=1.01)

# Average score distribution
ax = axes[0][0]
ax.hist(df["avg_score"], bins=30, color=PURPLE, alpha=0.8,
        edgecolor="white", linewidth=0.4)
ax.set_title("Average score distribution (new feature)", fontsize=11)
ax.set_xlabel("Average score"); ax.set_ylabel("Count")
sns.despine(ax=ax)

# Grade distribution
ax = axes[0][1]
gc = df["grade"].value_counts().sort_index()
bars = ax.bar(gc.index, gc.values, color=PAL[:len(gc)],
              edgecolor="white")
for bar, val in zip(bars, gc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(val), ha="center", fontsize=10)
ax.set_title("Grade distribution", fontsize=11)
ax.set_xlabel("Grade"); ax.set_ylabel("Count")
sns.despine(ax=ax)

# Avg score by parent education
ax = axes[1][0]
means = df.groupby("parent_education")["avg_score"].mean().reindex(edu_order)
bars  = ax.bar(range(len(means)), means.values, color=TEAL,
               edgecolor="white", alpha=0.85)
ax.set_xticks(range(len(means)))
ax.set_xticklabels(edu_order, rotation=20, ha="right", fontsize=9)
ax.set_title("Avg score by parent education", fontsize=11)
ax.set_ylabel("Mean avg score")
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f"{bar.get_height():.1f}", ha="center", fontsize=9)
sns.despine(ax=ax)

# Avg score by gender × test_prep
ax = axes[1][1]
grp = df.groupby(["gender","test_prep"])["avg_score"].mean().unstack()
grp.plot(kind="bar", ax=ax, color=[BLUE, CORAL], edgecolor="white",
         width=0.55)
ax.set_title("Avg score: gender × test prep", fontsize=11)
ax.set_xlabel(""); ax.set_ylabel("Mean avg score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Test Prep", fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
save("08_feature_engineering.png")

# ── Final Summary ──────────────────────────────────────────────
print(f"\n{'='*62}")
print("  EDA COMPLETE — KEY FINDINGS")
print(f"{'='*62}")
print(f"  Dataset         : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Missing values  : imputed with median (3 cols, ~5% each)")
print(f"  Outliers (IQR)  : "
      f"{sum(len(v) for v in iqr_outliers.values())} total across 3 score cols")
print(f"\n  Top correlations:")
for (c1, c2) in [("math_score","reading_score"),
                  ("math_score","wrting_score"),
                  ("reading_score","wrting_score")]:
    r = df[c1].corr(df[c2])
    print(f"    {c1} ↔ {c2}: r = {r:.3f}")
print(f"\n  Test prep impact on avg score:")
for prep, grp in df.groupby("test_prep")["avg_score"]:
    print(f"    {prep:12s}: mean = {grp.mean():.2f}")
print(f"\n  Parent education impact:")
for edu_lvl in edu_order:
    grp = df[df["parent_education"]==edu_lvl]["avg_score"]
    print(f"    {edu_lvl:20s}: mean = {grp.mean():.2f}")
print(f"\n  Outputs saved to: {OUTPUT}/")
print(f"{'='*62}")


── Creating derived features ──
   avg_score  total_score  score_range grade
0      76.89        230.7         16.8     B
1      87.37        262.1         24.0     A
2      66.26        198.8         17.0     C
3      57.54        172.6          6.3     C
4      49.03        147.1         16.9     D
5      39.20        117.6         13.2     F
6      48.09        144.3         13.1     D
7      56.60        169.8         11.0     C

── Grade distribution ──
grade
F     15
D    137
C    257
B     77
A     14
     Saved → 08_feature_engineering.png

  EDA COMPLETE — KEY FINDINGS
  Dataset         : 500 rows × 12 columns
  Missing values  : imputed with median (3 cols, ~5% each)
  Outliers (IQR)  : 31 total across 3 score cols

  Top correlations:
    math_score ↔ reading_score: r = 0.680
    math_score ↔ wrting_score: r = 0.687
    reading_score ↔ wrting_score: r = 0.562

  Test prep impact on avg score:
    completed   : mean = 64.84
    none        : mean = 58.01

  Parent education 